<a href="https://colab.research.google.com/github/mireillejb/FallDetectionSys/blob/main/Good_result_with_AM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ADVANCED LE2I FALL DETECTION - TARGET: 95%+ ACCURACY
# Goal: Accuracy≥95%, Precision≥90%, Recall=100%, Specificity≥95%, F1≥95%
# Strategy: Ensemble + Heavier Augmentation + Fine-tuned Weights

import os
import numpy as np
import cv2
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input, TimeDistributed, Conv1D, MaxPooling1D,
    Flatten, LSTM, Dense, Dropout, BatchNormalization, Add, Lambda, Layer,
    GlobalAveragePooling1D, Concatenate
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, Callback
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import random
from scipy.ndimage import gaussian_filter1d

print("TensorFlow version:", tf.__version__)

# ============================================================================
# REPRODUCIBILITY
# ============================================================================

def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    tf.config.experimental.enable_op_determinism()

set_all_seeds(42)

# ============================================================================
# DATASET DOWNLOAD
# ============================================================================

print("\n" + "="*70)
print("DOWNLOADING LE2I DATASET")
print("="*70)

import kagglehub
path = kagglehub.dataset_download("tuyenldvn/falldataset-imvia")
print("Path to dataset files:", path)

video_files = [os.path.join(dp, f) for dp, dn, fn in os.walk(path)
               for f in fn if f.endswith(".avi")]
print(f"✅ Found {len(video_files)} videos")

# ============================================================================
# MOVENET MODEL
# ============================================================================

print("\n" + "="*70)
print("LOADING MOVENET THUNDER MODEL")
print("="*70)

movenet = hub.load("https://tfhub.dev/google/movenet/singlepose/thunder/4")
print("✅ MoveNet loaded")

# ============================================================================
# KEYPOINT EXTRACTION (Keep your existing functions)
# ============================================================================

def extract_keypoints(image):
    img = tf.image.resize_with_pad(np.expand_dims(image, axis=0), 256, 256)
    img = tf.cast(img, dtype=tf.int32)
    outputs = movenet.signatures['serving_default'](img)
    keypoints = outputs['output_0'].numpy()
    return keypoints[0, 0, :, :]

def extract_sequence_from_video(video_path, max_frames=30):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return np.zeros((max_frames, 17, 3))
    if total_frames > max_frames:
        frame_indices = np.linspace(0, total_frames-1, max_frames, dtype=int)
    else:
        frame_indices = list(range(total_frames))
    frames_keypoints = []
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break
        kp = extract_keypoints(frame)
        frames_keypoints.append(kp)
    cap.release()
    while len(frames_keypoints) < max_frames:
        if len(frames_keypoints) > 0:
            frames_keypoints.append(frames_keypoints[-1])
        else:
            frames_keypoints.append(np.zeros((17, 3)))
    return np.array(frames_keypoints)

def read_annotation_file(annotation_path):
    try:
        with open(annotation_path, 'r') as f:
            lines = f.readlines()
        fall_start = int(lines[0].strip()) if len(lines) > 0 and lines[0].strip().isdigit() else 0
        fall_end = int(lines[1].strip()) if len(lines) > 1 and lines[1].strip().isdigit() else 0
        if fall_start == 0 and fall_end == 0:
            return 0
        overall_label = 0
        for line in lines[2:]:
            if line.strip():
                parts = line.strip().split(',')
                if len(parts) >= 6:
                    try:
                        label = int(parts[1])
                        if label > 0:
                            overall_label = 1
                    except ValueError:
                        continue
        return overall_label
    except:
        return 0

def find_annotation_file(video_path, dataset_path):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    possible_names = [f"{video_name}.txt", f"{video_name}_annotation.txt"]
    for root, dirs, files in os.walk(dataset_path):
        for possible_name in possible_names:
            if possible_name in files:
                return os.path.join(root, possible_name)
    return None

def extract_sequence_with_labels(video_path, annotation_path, max_frames=30):
    if not annotation_path or not os.path.exists(annotation_path):
        return extract_sequence_from_video(video_path, max_frames), None
    overall_label = read_annotation_file(annotation_path)
    sequence = extract_sequence_from_video(video_path, max_frames)
    return sequence, overall_label

def smart_label_assignment(video_files):
    import re
    labels = []
    for i, video_file in enumerate(video_files):
        filename = os.path.basename(video_file).lower()
        if any(word in filename for word in ['fall', 'falling', 'fell', 'chute']):
            label = 1
        elif any(word in filename for word in ['walk', 'normal', 'adl', 'sit', 'stand']):
            label = 0
        else:
            numbers = re.findall(r'\d+', filename)
            if numbers:
                video_num = int(numbers[0])
                label = 1 if video_num <= len(video_files) * 0.4 else 0
            else:
                label = 1 if i < len(video_files) * 0.4 else 0
        labels.append(label)
    return labels

# ============================================================================
# ADVANCED DATA AUGMENTATION (5x instead of 3x)
# ============================================================================

def augment_keypoint_sequence_advanced(sequence, augmentation_type='aggressive'):
    """Advanced augmentation with more variations"""
    augmented = sequence.copy()

    # Temporal shift
    if np.random.random() > 0.3:
        shift = np.random.randint(-3, 4)  # Increased range
        if shift > 0:
            augmented = np.concatenate([augmented[shift:], augmented[-shift:]], axis=0)
        elif shift < 0:
            augmented = np.concatenate([augmented[:shift], augmented[:abs(shift)]], axis=0)

    # Gaussian noise
    if np.random.random() > 0.2:
        noise = np.random.normal(0, 0.015, augmented.shape)  # Slightly more noise
        augmented = augmented + noise

    # Temporal smoothing
    if np.random.random() > 0.4:
        for i in range(augmented.shape[1]):
            for j in range(augmented.shape[2]):
                augmented[:, i, j] = gaussian_filter1d(augmented[:, i, j], sigma=np.random.uniform(0.3, 0.8))

    # Scale variation
    if np.random.random() > 0.3:
        scale = np.random.uniform(0.92, 1.08)  # Wider range
        augmented = augmented * scale

    # Random keypoint dropout
    if np.random.random() > 0.6:
        n_keypoints = augmented.shape[1]
        dropout_idx = np.random.choice(n_keypoints, size=np.random.randint(1, 4), replace=False)
        augmented[:, dropout_idx, :] = 0

    # Random frame dropout
    if np.random.random() > 0.7:
        frame_dropout = np.random.choice(augmented.shape[0], size=np.random.randint(1, 3), replace=False)
        if len(frame_dropout) < augmented.shape[0] - 5:  # Keep at least some frames
            for fd in frame_dropout:
                if fd > 0:
                    augmented[fd] = augmented[fd-1]

    return augmented

def create_heavily_augmented_dataset(X, y, augmentation_factor=5):
    """Create heavily augmented dataset - augment BOTH classes"""
    X_augmented = []
    y_augmented = []

    class_indices = np.argmax(y, axis=1)

    # Original samples
    for i in range(len(X)):
        X_augmented.append(X[i])
        y_augmented.append(y[i])

    # Augment both classes, but more for minority
    for i in range(len(X)):
        if class_indices[i] == 0:  # Normal - if minority
            n_aug = augmentation_factor + 2  # Extra augmentation
        else:  # Falls
            n_aug = augmentation_factor

        for _ in range(n_aug):
            aug_seq = augment_keypoint_sequence_advanced(X[i])
            X_augmented.append(aug_seq)
            y_augmented.append(y[i])

    X_augmented = np.array(X_augmented)
    y_augmented = np.array(y_augmented)

    indices = np.arange(len(X_augmented))
    np.random.shuffle(indices)
    X_augmented = X_augmented[indices]
    y_augmented = y_augmented[indices]

    print(f"\n📊 Heavy Augmentation:")
    print(f"  Original: {len(X)} → Augmented: {len(X_augmented)} ({len(X_augmented)/len(X):.1f}x)")
    aug_class_counts = np.bincount(np.argmax(y_augmented, axis=1))
    print(f"  Normal: {aug_class_counts[0]} ({aug_class_counts[0]/len(y_augmented)*100:.1f}%)")
    print(f"  Fall: {aug_class_counts[1]} ({aug_class_counts[1]/len(y_augmented)*100:.1f}%)")

    return X_augmented, y_augmented

# ============================================================================
# DATASET PROCESSING
# ============================================================================

def process_le2i_dataset(video_files, labels, max_frames=30, val_size=0.22):
    X, y = [], []
    print("\n" + "="*70)
    print("PROCESSING LE2I VIDEOS")
    print("="*70)

    for i, (video_path, heuristic_label) in enumerate(tqdm(zip(video_files, labels),
                                                            total=len(video_files),
                                                            desc="Extracting keypoints")):
        annotation_path = find_annotation_file(video_path, os.path.dirname(os.path.dirname(video_path)))
        sequence, annotation_label = extract_sequence_with_labels(video_path, annotation_path, max_frames)

        if sequence is not None and sequence.shape[0] > 0:
            X.append(sequence)
            y.append(annotation_label if annotation_label is not None else heuristic_label)

    X = np.array(X)
    y = np.array(y)
    y_categorical = to_categorical(y, num_classes=2)

    fall_count = np.sum(y)
    normal_count = len(y) - fall_count

    print(f"\n✅ Dataset processed: {X.shape[0]} videos")
    print(f"   Fall: {fall_count} ({fall_count/len(y)*100:.1f}%)")
    print(f"   Normal: {normal_count} ({normal_count/len(y)*100:.1f}%)")

    if len(np.unique(y)) > 1:
        try:
            X_train, X_val, y_train, y_val = train_test_split(
                X, y_categorical, test_size=val_size, stratify=y, random_state=42
            )
        except:
            X_train, X_val, y_train, y_val = train_test_split(
                X, y_categorical, test_size=val_size, random_state=42
            )
    else:
        X_train, X_val, y_train, y_val = train_test_split(
            X, y_categorical, test_size=val_size, random_state=42
        )

    print(f"   Training: {X_train.shape[0]}, Validation: {X_val.shape[0]}")
    return X_train, X_val, y_train, y_val

# ============================================================================
# CUSTOM LAYERS (Keep your existing implementations)
# ============================================================================

class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, alpha=0.25, gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        self.alpha = alpha
        self.gamma = gamma
    def call(self, y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        ce = -y_true * tf.math.log(y_pred)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        focal_term = tf.pow(1. - pt, self.gamma)
        focal_loss = self.alpha * focal_term * ce
        return tf.reduce_sum(focal_loss, axis=-1)
    def get_config(self):
        config = super().get_config()
        config.update({"alpha": self.alpha, "gamma": self.gamma})
        return config

class TemporalSelfAttention(Layer):
    def __init__(self, d_model, num_heads=4, dropout_rate=0.25, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.dropout_rate = dropout_rate
        assert d_model % num_heads == 0
        self.depth = d_model // num_heads
        self.wq = Dense(d_model, kernel_regularizer=l2(0.0005))
        self.wk = Dense(d_model, kernel_regularizer=l2(0.0005))
        self.wv = Dense(d_model, kernel_regularizer=l2(0.0005))
        self.dense = Dense(d_model, kernel_regularizer=l2(0.0005))
        self.dropout = Dropout(dropout_rate)
        self.layer_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def scaled_dot_product_attention(self, q, k, v, mask=None):
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)
        if mask is not None:
            scaled_attention_logits += (mask * -1e9)
        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        attention_weights = self.dropout(attention_weights)
        output = tf.matmul(attention_weights, v)
        return output, attention_weights

    def call(self, inputs, training=None, mask=None):
        batch_size = tf.shape(inputs)[0]
        seq_len = tf.shape(inputs)[1]
        q = self.wq(inputs)
        k = self.wk(inputs)
        v = self.wv(inputs)
        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)
        scaled_attention, _ = self.scaled_dot_product_attention(q, k, v, mask)
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(scaled_attention, (batch_size, seq_len, self.d_model))
        output = self.dense(concat_attention)
        output = self.layer_norm(inputs + output)
        return output

    def get_config(self):
        config = super().get_config()
        config.update({'d_model': self.d_model, 'num_heads': self.num_heads, 'dropout_rate': self.dropout_rate})
        return config

class BalancedMetricsCallback(Callback):
    def __init__(self, validation_data):
        super().__init__()
        self.validation_data = validation_data
        self.best_balanced_score = 0

    def on_epoch_end(self, epoch, logs=None):
        x_val, y_val = self.validation_data
        y_pred = self.model.predict(x_val, verbose=0)
        y_pred_classes = np.argmax(y_pred, axis=1)
        y_true_classes = np.argmax(y_val, axis=1)
        cm = confusion_matrix(y_true_classes, y_pred_classes)
        tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        balanced_score = (precision + recall + specificity) / 3
        print(f"\n[Epoch {epoch + 1}] P: {precision:.4f}, R: {recall:.4f}, S: {specificity:.4f}, Bal: {balanced_score:.4f}")
        print(f"              FP: {fp}, FN: {fn}, TP: {tp}, TN: {tn}")
        if balanced_score > self.best_balanced_score:
            self.best_balanced_score = balanced_score
            print(f"              ✓ New best!")

def normalize_keypoints(x):
    mean = tf.reduce_mean(x, axis=1, keepdims=True)
    std = tf.math.reduce_std(x, axis=1, keepdims=True) + 1e-8
    return (x - mean) / std

# ============================================================================
# OPTIMIZED MODEL (Reduced dropout even more)
# ============================================================================

def create_high_performance_model(n_steps, n_features):
    inputs = Input(shape=(n_steps, n_features, 3))
    x = Lambda(normalize_keypoints)(inputs)

    # CNN - Very low dropout for better learning
    x = TimeDistributed(Conv1D(64, 3, padding='same', activation="relu", kernel_regularizer=l2(0.0005)))(x)
    x = TimeDistributed(BatchNormalization())(x)
    x = TimeDistributed(Dropout(0.10))(x)  # Reduced

    residual1 = x
    x = TimeDistributed(Conv1D(64, 3, padding='same', activation="relu", kernel_regularizer=l2(0.0005)))(x)
    x = TimeDistributed(BatchNormalization())(x)
    x = Add()([x, residual1])
    x = TimeDistributed(MaxPooling1D(2))(x)
    x = TimeDistributed(Dropout(0.15))(x)  # Reduced

    x = TimeDistributed(Conv1D(128, 3, padding='same', activation="relu", kernel_regularizer=l2(0.0005)))(x)
    x = TimeDistributed(BatchNormalization())(x)
    x = TimeDistributed(Dropout(0.15))(x)  # Reduced

    x = TimeDistributed(Conv1D(128, 3, padding='same', activation="relu", kernel_regularizer=l2(0.0005)))(x)
    x = TimeDistributed(BatchNormalization())(x)
    x = TimeDistributed(MaxPooling1D(2))(x)
    x = TimeDistributed(Dropout(0.20))(x)  # Reduced

    x = TimeDistributed(Conv1D(160, 3, padding='same', activation="relu", kernel_regularizer=l2(0.0005)))(x)
    x = TimeDistributed(BatchNormalization())(x)
    x = TimeDistributed(Dropout(0.20))(x)  # Reduced

    x = TimeDistributed(Flatten())(x)

    # BiLSTM with Attention
    lstm_out = tf.keras.layers.Bidirectional(
        LSTM(128, return_sequences=True, kernel_regularizer=l2(0.0002),
             recurrent_regularizer=l2(0.0002), dropout=0.20, recurrent_dropout=0.15),
        merge_mode='concat')(x)

    attention_out = TemporalSelfAttention(d_model=256, num_heads=4, dropout_rate=0.20)(lstm_out)

    path1 = tf.keras.layers.Bidirectional(
        LSTM(64, kernel_regularizer=l2(0.0002), recurrent_regularizer=l2(0.0002),
             dropout=0.25, recurrent_dropout=0.20), merge_mode='concat')(attention_out)

    path2 = GlobalAveragePooling1D()(lstm_out)
    combined = Concatenate()([path1, path2])

    # Dense layers
    x = Dense(128, activation="relu", kernel_regularizer=l2(0.002))(combined)
    x = BatchNormalization()(x)
    x = Dropout(0.25)(x)  # Reduced

    x = Dense(64, activation="relu", kernel_regularizer=l2(0.002))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.20)(x)  # Reduced

    outputs = Dense(2, activation="softmax")(x)

    model = Model(inputs=inputs, outputs=outputs)
    optimizer = Adam(learning_rate=0.0003, clipnorm=1.0)  # Lower LR for stability
    model.compile(
        optimizer=optimizer,
        loss=FocalLoss(alpha=0.25, gamma=2.0),
        metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
                tf.keras.metrics.Recall(name='recall')]
    )
    return model

# ============================================================================
# TRAINING WITH OPTIMIZED WEIGHTS
# ============================================================================

def train_high_performance_model(x_train, y_train, x_val, y_val, epochs=120):
    print("\n" + "="*70)
    print("HIGH-PERFORMANCE TRAINING - TARGET: 95%+ METRICS")
    print("="*70)

    model = create_high_performance_model(30, 17)
    print("\n✓ Model created")

    # Optimized class weights to maximize recall while maintaining specificity
    class_weights = {
        0: 2.8,  # Normal - slightly increased to reduce FP
        1: 4.5   # Fall - increased to minimize FN (target: 100% recall)
    }

    print(f"\nClass Weights: Normal={class_weights[0]}, Fall={class_weights[1]}")

    callbacks = [
        BalancedMetricsCallback(validation_data=(x_val, y_val)),
        EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, min_delta=0.0005, verbose=1),
        ModelCheckpoint('best_model_95percent.keras', monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=12, min_lr=1e-7, verbose=1)
    ]

    history = model.fit(
        x_train, y_train,
        batch_size=16,
        epochs=epochs,
        validation_data=(x_val, y_val),
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1,
        shuffle=True
    )

    return model, history

# ============================================================================
# DETAILED EVALUATION
# ============================================================================

def evaluate_detailed(model, x_val, y_val):
    print("\n" + "="*70)
    print("FINAL EVALUATION - TARGET METRICS CHECK")
    print("="*70)

    y_pred = model.predict(x_val, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_val, axis=1)

    cm = confusion_matrix(y_true_classes, y_pred_classes)
    tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]

    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print("\n📊 CONFUSION MATRIX:")
    print(f"   TN: {tn:2d}  FP: {fp:2d}")
    print(f"   FN: {fn:2d}  TP: {tp:2d}")

    print("\n🎯 PERFORMANCE METRICS:")
    print(f"   • Accuracy:     {accuracy:.4f} ({accuracy*100:.2f}%)  [Target: ≥95%]  {'✓' if accuracy >= 0.95 else '✗'}")
    print(f"   • Precision:    {precision:.4f} ({precision*100:.2f}%)  [Target: ≥90%]  {'✓' if precision >= 0.90 else '✗'}")
    print(f"   • Recall:       {recall:.4f} ({recall*100:.2f}%)  [Target: =100%] {'✓' if recall == 1.0 else '✗'}")
    print(f"   • Specificity:  {specificity:.4f} ({specificity*100:.2f}%)  [Target: ≥95%]  {'✓' if specificity >= 0.95 else '✗'}")
    print(f"   • F1-Score:     {f1:.4f} ({f1*100:.2f}%)  [Target: ≥95%]  {'✓' if f1 >= 0.95 else '✗'}")

    targets_met = 0
    if accuracy >= 0.95: targets_met += 1
    if precision >= 0.90: targets_met += 1
    if recall == 1.0: targets_met += 1
    if specificity >= 0.95: targets_met += 1
    if f1 >= 0.95: targets_met += 1

    print(f"\n✅ TARGETS MET: {targets_met}/5")

    if targets_met < 5:
        print("\n💡 RECOMMENDATIONS:")
        if recall < 1.0:
            print(f"   → Recall is {recall:.2%}. Increase fall_weight to 5.0-5.5")
        if specificity < 0.95:
            print(f"   → Specificity is {specificity:.2%}. Increase normal_weight to 3.0-3.5")
        if accuracy < 0.95:
            print("   → More augmentation or longer training may help")

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'f1': f1,
        'confusion_matrix': cm,
        'targets_met': targets_met
    }

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("\n" + "="*70)
    print("ADVANCED LE2I FALL DETECTION - TARGET: 95%+ METRICS")
    print("="*70)
    print("\n🎯 TARGETS:")
    print("   • Accuracy:     ≥ 95%")
    print("   • Precision:    ≥ 90%")
    print("   • Recall:       = 100%")
    print("   • Specificity:  ≥ 95%")
    print("   • F1-Score:     ≥ 95%")

    # Process dataset
    labels = smart_label_assignment(video_files)
    X_train, X_val, y_train, y_val = process_le2i_dataset(
        video_files, labels, max_frames=30, val_size=0.22
    )

    if X_train is None:
        print("❌ ERROR: Dataset processing failed!")
        exit(1)

    # Heavy augmentation (5x + extra for minority class)
    print("\n" + "="*70)
    print("APPLYING HEAVY AUGMENTATION (5x)")
    print("="*70)
    X_train_aug, y_train_aug = create_heavily_augmented_dataset(X_train, y_train, augmentation_factor=5)

    # Train
    print("\n⚙️ CONFIGURATION:")
    print("  • Focal Loss: Yes")
    print("  • Heavy Augmentation: 5-7x")
    print("  • Normal Weight: 2.8")
    print("  • Fall Weight: 4.5")
    print("  • Learning Rate: 0.0003")
    print("  • Epochs: 120")
    print("  • Dropout: 0.10-0.25 (very low)")

    model, history = train_high_performance_model(
        X_train_aug, y_train_aug, X_val, y_val, epochs=120
    )

    # Evaluate
    results = evaluate_detailed(model, X_val, y_val)

    print("\n" + "="*70)
    print("✅ TRAINING COMPLETED!")
    print("="*70)
    print(f"\n💾 Best model saved as: best_model_95percent.keras")
    print(f"🎯 Targets met: {results['targets_met']}/5")

    if results['targets_met'] == 5:
        print("\n🎉 CONGRATULATIONS! ALL TARGETS ACHIEVED!")
    else:
        print("\n📝 If targets not met, try:")
        print("  1. Increase epochs to 150-200")
        print("  2. Adjust weights: fall_weight=5.0-5.5 for higher recall")
        print("  3. Increase augmentation_factor to 7-10")
        print("  4. Train multiple models and use ensemble")

    print("\n" + "="*70)

TensorFlow version: 2.19.0

DOWNLOADING LE2I DATASET


100%|██████████| 9.37G/9.37G [01:43<00:00, 97.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/tuyenldvn/falldataset-imvia/versions/2
✅ Found 190 videos

LOADING MOVENET THUNDER MODEL
✅ MoveNet loaded

ADVANCED LE2I FALL DETECTION - TARGET: 95%+ METRICS

🎯 TARGETS:
   • Accuracy:     ≥ 95%
   • Precision:    ≥ 90%
   • Recall:       = 100%
   • Specificity:  ≥ 95%
   • F1-Score:     ≥ 95%

PROCESSING LE2I VIDEOS


Extracting keypoints: 100%|██████████| 190/190 [12:22<00:00,  3.91s/it]



✅ Dataset processed: 190 videos
   Fall: 156 (82.1%)
   Normal: 34 (17.9%)
   Training: 148, Validation: 42

APPLYING HEAVY AUGMENTATION (5x)

📊 Heavy Augmentation:
  Original: 148 → Augmented: 940 (6.4x)
  Normal: 208 (22.1%)
  Fall: 732 (77.9%)

⚙️ CONFIGURATION:
  • Focal Loss: Yes
  • Heavy Augmentation: 5-7x
  • Normal Weight: 2.8
  • Fall Weight: 4.5
  • Learning Rate: 0.0003
  • Epochs: 120
  • Dropout: 0.10-0.25 (very low)

HIGH-PERFORMANCE TRAINING - TARGET: 95%+ METRICS

✓ Model created

Class Weights: Normal=2.8, Fall=4.5
Epoch 1/120
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5294 - loss: 2.4037 - precision: 0.5294 - recall: 0.5294
[Epoch 1] P: 0.8205, R: 0.9412, S: 0.1250, Bal: 0.6289
              FP: 7, FN: 2, TP: 32, TN: 1
              ✓ New best!

Epoch 1: val_accuracy improved from -inf to 0.78571, saving model to best_model_95percent.keras
59/59 ━━━━━━━━━━━━━━━━━━━━ 231s 2s/step - accuracy: 0.5302 - loss: 2.4014 - precision: 0.5302 - recall: 0.5302 - val_ac